In [ ]:
!pip install hopsworks

In [ ]:
!pip install confluent-kafka

In [ ]:
from google.colab import files
uploaded = files.upload()  # apni feature_df.csv choose karo

Saving feature_df.csv to feature_df (1).csv


In [ ]:
import getpass
HOPSWORKS_API_KEY = getpass.getpass("Enter your Hopsworks API key: ")
HOPSWORKS_PROJECT_NAME = input("Enter your Hopsworks project name: ")


Enter your Hopsworks API key: ··········
Enter your Hopsworks project name: practice_project


In [ ]:
import hopsworks
import pandas as pd

# Load and prepare
df = pd.read_csv("feature_df.csv")
df["collection_timestamp"] = pd.to_datetime(df["collection_timestamp"], utc=True).dt.tz_localize(None)
if "timestamp" in df.columns:
    df = df.drop(columns=["timestamp"])

# Login
project = hopsworks.login(project=HOPSWORKS_PROJECT_NAME, api_key_value=HOPSWORKS_API_KEY)
fs = project.get_feature_store()

# Create/get feature group
fg = fs.get_or_create_feature_group(
    name="aqi_features",
    version=1,
    description="Engineered AQI + weather features (Phase 2 output).",
    primary_key=["city", "collection_timestamp"],
    event_time="collection_timestamp",
    online_enabled=False,
    time_travel_format="HUDI",
)

# Insert
fg.insert(df)
print(f"Inserted {len(df)} rows.")

# Read back to confirm
result = fg.read()
print(f"Read back {len(result)} rows.")
print(result.head())


Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41109


Uploading Dataframe: 100.00% |██████████| Rows 1936/1936 | Elapsed Time: 00:02 | Remaining Time: 00:00


Launching job: aqi_features_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/41109/jobs/named/aqi_features_1_offline_fg_materialization/executions
Inserted 1936 rows.


ERROR:hsfs.core.arrow_flight_client:No hudi properties found for featuregroup: /apps/hive/warehouse/practice_project_featurestore.db/aqi_features_1 - This usually means that no data has been written yet to this feature group. Detail: Failed. gRPC client debug context: UNKNOWN:Error received from peer ipv4:57.130.64.132:5005 {created_time:"2026-07-26T20:46:00.099406273+00:00", grpc_status:2, grpc_message:"No hudi properties found for featuregroup: /apps/hive/warehouse/practice_project_featurestore.db/aqi_features_1 - This usually means that no data has been written yet to this feature group. Detail: Failed"}. Client context: IOError: Server never sent a data message. Detail: Internal
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/hsfs/core/arrow_flight_client.py", line 433, in afs_error_handler_wrapper
    return func(instance, *args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/hsfs/core/arrow_flight_cl

Error: Reading data from Hopsworks, using Hopsworks Feature Query Service           


FeatureStoreException: No hudi properties found for featuregroup: /apps/hive/warehouse/practice_project_featurestore.db/aqi_features_1 - This usually means that no data has been written yet to this feature group. Detail: Failed. gRPC client debug context: UNKNOWN:Error received from peer ipv4:57.130.64.132:5005 {created_time:"2026-07-26T20:46:00.099406273+00:00", grpc_status:2, grpc_message:"No hudi properties found for featuregroup: /apps/hive/warehouse/practice_project_featurestore.db/aqi_features_1 - This usually means that no data has been written yet to this feature group. Detail: Failed"}. Client context: IOError: Server never sent a data message. Detail: Internal

In [ ]:
result = fg.read()
print(f"Read back {len(result)} rows.")
print(result.head())

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.06s) 
Read back 1936 rows.
       collection_timestamp    city   latitude  longitude  temperature  \
0 2026-03-03 06:00:00+00:00  Sukkur  27.732864  68.865166         28.8   
1 2026-06-07 06:00:00+00:00  Sukkur  27.732864  68.865166         38.4   
2 2026-04-11 03:00:00+00:00  Sukkur  27.732864  68.865166         24.1   
3 2026-02-17 03:00:00+00:00  Sukkur  27.732864  68.865166         15.9   
4 2026-07-17 18:00:00+00:00  Sukkur  27.732864  68.865166         34.6   

   feels_like  humidity  pressure  wind_speed  wind_direction  ...  \
0         NaN        46    1006.5         4.0             267  ...   
1         NaN        43     994.9         7.9             145  ...   
2         NaN        59    1003.9        10.2              42  ...   
3         NaN        88    1004.3        19.5             299  ...   
4         NaN        47     991.6        19.3             196  ...   

   pm2_5_rolling_mean_24h  

In [ ]:
df["pm10"].isna().sum()

np.int64(1936)

TypeError: Object of type DataFrame is not JSON serializable